In [85]:

import numpy as np
from pathlib import Path

project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

# Load 3D sensor sequences directly
data = np.load(
    project_dir / "results" / "raw_window_sequences_split.npz"
)

X_train_lstm = data["X_train"]
X_test_lstm = data["X_test"]

y_train_lstm = data["y_train"]
y_test_lstm = data["y_test"]

print("Training:", X_train_lstm.shape)
print("Testing:", X_test_lstm.shape)
print("Training labels:", y_train_lstm.shape)
print("Testing labels:", y_test_lstm.shape)


Training: (500, 360, 5)
Testing: (127, 360, 5)
Training labels: (500,)
Testing labels: (127,)



### Scale Sensor Values for LSTM


In [86]:

# Calculate scaling statistics using training data only
sensor_median = np.median(X_train_lstm, axis=(0, 1))
sensor_q25 = np.percentile(X_train_lstm, 25, axis=(0, 1))
sensor_q75 = np.percentile(X_train_lstm, 75, axis=(0, 1))

# Calculate Interquartile Range (IQR)
sensor_iqr = sensor_q75 - sensor_q25
sensor_iqr = np.where(sensor_iqr == 0, 1, sensor_iqr)

# Scale training data
X_train_scaled = (X_train_lstm - sensor_median) / sensor_iqr

# Scale testing data using training statistics
X_test_scaled = (X_test_lstm - sensor_median) / sensor_iqr

# Limit extreme sensor values
X_train_scaled = np.clip(X_train_scaled, -5, 5).astype(np.float32)
X_test_scaled = np.clip(X_test_scaled, -5, 5).astype(np.float32)

print("Training shape:", X_train_scaled.shape)
print("Testing shape:", X_test_scaled.shape)
print("Training missing values:", np.isnan(X_train_scaled).sum())
print("Testing missing values:", np.isnan(X_test_scaled).sum())
print("Training value range:", X_train_scaled.min(), X_train_scaled.max())


Training shape: (500, 360, 5)
Testing shape: (127, 360, 5)
Training missing values: 0
Testing missing values: 0
Training value range: -5.0 5.0



### Define LSTM Model Architecture


In [ ]:


import torch
from torch import nn

class FatigueLSTM(nn.Module):
    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=5,
            hidden_size=32,
            num_layers=1,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        _, (hidden, _) = self.lstm(x)

        x = hidden[-1]
        x = self.dropout(x)

        return self.fc(x).squeeze(-1)


# Checking for GPU availability
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Initialize model
lstm_model = FatigueLSTM().to(device)

print(lstm_model)
print("\nDevice:", device)
print("Total parameters:", sum(
    p.numel() for p in lstm_model.parameters()
))



FatigueLSTM(
  (lstm): LSTM(5, 32, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)

Device: cuda
Total parameters: 5025


In [88]:
import torch

print("PyTorch version:", torch.__version__)
print("PyTorch CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.14.1+cu126
PyTorch CUDA version: 12.6
CUDA available: True


In [89]:

import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.14.1+cu126
CUDA: 12.6
GPU available: True
GPU: NVIDIA GeForce RTX 2050



### Initialize LSTM on GPU


In [90]:

# Select NVIDIA GPU
device = torch.device("cuda")

# Move LSTM model to GPU
lstm_model = lstm_model.to(device)

# Verify model is on GPU
print("Model device:", next(lstm_model.parameters()).device)
print("GPU:", torch.cuda.get_device_name(0))
print("Training data shape:", X_train_scaled.shape)


Model device: cuda:0
GPU: NVIDIA GeForce RTX 2050
Training data shape: (500, 360, 5)



### Prepare LSTM DataLoaders


In [91]:

from torch.utils.data import TensorDataset, DataLoader

# Convert NumPy arrays into PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_lstm, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_lstm, dtype=torch.float32)

# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

# Verify batch shapes
batch_X, batch_y = next(iter(train_loader))

print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))
print("Batch input shape:", batch_X.shape)
print("Batch target shape:", batch_y.shape)


Training batches: 32
Testing batches: 8
Batch input shape: torch.Size([16, 360, 5])
Batch target shape: torch.Size([16])



### Configure LSTM Loss Function and Optimizer


In [92]:

# Define loss function
criterion = nn.MSELoss()

# Define optimizer
optimizer = torch.optim.Adam(
    lstm_model.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

print("Loss Function:", criterion)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning Rate:", optimizer.param_groups[0]["lr"])
print("Model Device:", next(lstm_model.parameters()).device)


Loss Function: MSELoss()
Optimizer: Adam
Learning Rate: 0.001
Model Device: cuda:0



### Train LSTM Model


In [93]:

# Training configuration
epochs = 30
train_losses = []

for epoch in range(epochs):

    lstm_model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:

        # Move data to GPU
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        # Reset gradients
        optimizer.zero_grad()

        # Predict fatigue scores
        predictions = lstm_model(batch_X)

        # Calculate loss
        loss = criterion(predictions, batch_y)

        # Update model weights
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            lstm_model.parameters(), max_norm=1.0
        )
        optimizer.step()

        total_loss += loss.item() * batch_X.size(0)

    # Average training loss
    epoch_loss = total_loss / len(train_dataset)
    train_losses.append(epoch_loss)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(
            f"Epoch [{epoch + 1}/{epochs}] "
            f"Training Loss: {epoch_loss:.4f}"
        )

print("\nLSTM training completed!")


Epoch [1/30] Training Loss: 21.9400
Epoch [5/30] Training Loss: 5.5541
Epoch [10/30] Training Loss: 5.2481
Epoch [15/30] Training Loss: 4.9098
Epoch [20/30] Training Loss: 4.8966
Epoch [25/30] Training Loss: 4.9081
Epoch [30/30] Training Loss: 4.7934

LSTM training completed!



### Evaluate LSTM Regression


In [94]:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Switch model to evaluation mode
lstm_model.eval()

# Predict using the test dataset
with torch.no_grad():
    predictions = lstm_model(
        torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
    )

# Convert predictions to NumPy
y_pred_lstm = predictions.cpu().numpy()

# Keep fatigue scores between 0 and 10
y_pred_lstm = np.clip(y_pred_lstm, 0, 10)

# Calculate regression metrics
lstm_mae = mean_absolute_error(y_test_lstm, y_pred_lstm)
lstm_mse = mean_squared_error(y_test_lstm, y_pred_lstm)
lstm_rmse = np.sqrt(lstm_mse)
lstm_r2 = r2_score(y_test_lstm, y_pred_lstm)

print(f"MAE: {lstm_mae:.4f}")
print(f"MSE: {lstm_mse:.4f}")
print(f"RMSE: {lstm_rmse:.4f}")
print(f"R² Score: {lstm_r2:.4f}")


MAE: 2.1427
MSE: 5.7521
RMSE: 2.3984
R² Score: 0.0279



### Evaluate LSTM Classification Accuracy


In [95]:

from sklearn.metrics import accuracy_score

# Convert fatigue scores into Low, Moderate, High
y_test_lstm_levels = np.digitize(y_test_lstm, bins=[4, 7])
y_pred_lstm_levels = np.digitize(y_pred_lstm, bins=[4, 7])

# Calculate classification accuracy
lstm_accuracy = accuracy_score(
    y_test_lstm_levels,
    y_pred_lstm_levels
)

print(f"LSTM Accuracy: {lstm_accuracy * 100:.2f}%")


LSTM Accuracy: 37.80%



### Evaluate LSTM High Fatigue Recall


In [96]:

# Identify actual High Fatigue windows
high_mask_lstm = y_test_lstm_levels == 2

# Count actual and correctly detected windows
total_high_lstm = high_mask_lstm.sum()

detected_high_lstm = (
    y_pred_lstm_levels[high_mask_lstm] == 2
).sum()

# Calculate High Fatigue Recall
lstm_high_recall = (
    detected_high_lstm / total_high_lstm
    if total_high_lstm > 0 else 0
)

print("Actual High Fatigue Windows:", total_high_lstm)
print("Correctly Detected:", detected_high_lstm)
print(f"High Fatigue Recall: {lstm_high_recall * 100:.2f}%")


Actual High Fatigue Windows: 19
Correctly Detected: 0
High Fatigue Recall: 0.00%



### Save LSTM 3-Minute Window Sequence Model


In [97]:

# Create model directory
model_dir = project_dir / "models"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "lstm_3min_sequence.pth"

# Save model weights and preprocessing information
torch.save({
    "model_state_dict": lstm_model.state_dict(),
    "input_size": 5,
    "hidden_size": 32,
    "num_layers": 1,
    "window_size": 360,
    "sensor_median": torch.tensor(sensor_median),
    "sensor_iqr": torch.tensor(sensor_iqr),
    "clip_range": (-5, 5),
    "thresholds": (4, 7)
}, model_path)

print("LSTM model saved successfully!")
print("Location:", model_path)


LSTM model saved successfully!
Location: c:\Users\WINSYS\Desktop\Physical_Fatigue_ML\models\lstm_3min_sequence.pth
